# 19 - Qwen3-Reranker-8B LoRA Fine-tuning and Evaluation

Fine-tune `Qwen/Qwen3-Reranker-8B` with leakage-checked legal query-passage labels, then evaluate it on Qwen3 dense top-30 candidates.


In [ ]:
!pip install -q -U "torchao>=0.16.0" "peft>=0.15.0" "transformers>=4.51.0" "sentence-transformers>=3.0.0" accelerate bitsandbytes faiss-cpu rank-bm25 tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))

# Colab can keep old project modules in memory after files are replaced in Drive.
# Clear them so the updated reranking/finetune code is imported.
for module_name in [
    'src.reranking',
    'src.evaluation_reranker',
    'src.finetune_reranker_model',
    'reranking',
    'evaluation_reranker',
    'finetune_reranker_model',
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


In [ ]:
from src.finetune_reranker_model import finetune_reranker_model

tuned_reranker_dir = DRIVE_ROOT / 'models/reranker_tuned/qwen3_reranker_8b_legal_lora_v5'
adapter_config = tuned_reranker_dir / 'adapter_config.json'
adapter_model_safe = tuned_reranker_dir / 'adapter_model.safetensors'
adapter_model_bin = tuned_reranker_dir / 'adapter_model.bin'

if adapter_config.exists() and (adapter_model_safe.exists() or adapter_model_bin.exists()):
    print(f'Existing tuned reranker adapter found, skipping training: {tuned_reranker_dir}')
    config_path = tuned_reranker_dir / 'reranker_finetune_config.json'
    run_config = json.loads(config_path.read_text(encoding='utf-8')) if config_path.exists() else {'output_dir': str(tuned_reranker_dir)}
else:
    run_config = finetune_reranker_model(
        train_jsonl=DRIVE_ROOT / 'data/processed/reranker_tune_train.jsonl',
        val_jsonl=DRIVE_ROOT / 'data/processed/reranker_tune_val.jsonl',
        output_dir=tuned_reranker_dir,
        model_name='Qwen/Qwen3-Reranker-8B',
        max_train_samples=None,
        max_val_samples=None,
        batch_size=1,
        epochs=1,
        learning_rate=1e-5,
        max_length=2048,
        use_lora=True,
        lora_r=32,
        lora_alpha=64,
        lora_dropout=0.05,
        device=device,
    )

print(tuned_reranker_dir)
print([p.name for p in tuned_reranker_dir.iterdir()])
run_config


In [ ]:
from src.reranking import CrossEncoderReranker

# Quick load check before the 190-question evaluation.
reranker_load_check = CrossEncoderReranker(model_name=str(tuned_reranker_dir), device=device)
print(type(reranker_load_check).__name__)
print(getattr(reranker_load_check, 'model_name', str(tuned_reranker_dir)))
del reranker_load_check
torch.cuda.empty_cache() if device == 'cuda' else None


In [ ]:
from src.evaluation_reranker import evaluate_reranker

index_root = DRIVE_ROOT / config['best_retrieval']['index_root']
summary = evaluate_reranker(
    benchmark_csv=DRIVE_ROOT / config['benchmark_csv'],
    index_root=index_root,
    output_predictions_csv=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_top30_qwen3_reranker_8b_legal_tuned_predictions_v1.csv',
    output_summary_json=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_top30_qwen3_reranker_8b_legal_tuned_summary_v1.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model=str(tuned_reranker_dir),
    batch_size=4,
    device=device,
)
summary


## Optional Combined Tuned Embedding + Tuned Reranker Evaluation

Run this after notebook 18 has created the tuned embedding index. This measures the fully tuned retrieval stack before any expensive generation rerun.

In [ ]:
from src.evaluation_reranker import evaluate_reranker

tuned_embedding_index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b_legal_tuned'
if not tuned_embedding_index_root.exists():
    raise FileNotFoundError(f'Tuned embedding index not found. Run notebook 18 first: {tuned_embedding_index_root}')

combined_summary = evaluate_reranker(
    benchmark_csv=DRIVE_ROOT / config['benchmark_csv'],
    index_root=tuned_embedding_index_root,
    output_predictions_csv=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_legal_tuned_dense_top30_qwen3_reranker_8b_legal_tuned_predictions_v1.csv',
    output_summary_json=DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_legal_tuned_dense_top30_qwen3_reranker_8b_legal_tuned_summary_v1.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model=str(tuned_reranker_dir),
    batch_size=4,
    device=device,
)
combined_summary
